# Imports

In [1]:
import pandas as pd
import evaluate
from datasets import load_dataset

# Load data

In [2]:
# Load the revised dataset
data_revised = pd.read_csv('data/AWAL evaluation sets - FLORES devtest (REVISED).csv')

data_revised["ID"] = data_revised.index

# remove unnecessary columns
data_revised = data_revised[["ID", "English", "Tamazight (Corrected)"]]
data_revised.columns = ["ID", "eng", "zgh"]

data_revised.describe(include='all')

,ID,eng,zgh
count,1012.000000,1012,1012
unique,NaN,1012,1012
top,NaN,Workers must often get their superiors' approv...,ⵉⴳⴳⵓⴷⵉ ⵎⴰ ⴳ ⴷ ⵉⵇⵇⴰⵏ ⴰⴷ ⵢⵉⵍⵉ ⵓⵎⵙⴰⵙⴰ ⵏ ⵉⵏⵙⵙⵉⵅⴼⵏ ...
freq,NaN,1,1
mean,505.500000,NaN,NaN
std,292.283538,NaN,NaN
min,0.000000,NaN,NaN
25%,252.750000,NaN,NaN
50%,505.500000,NaN,NaN
75%,758.250000,NaN,NaN


In [3]:
# Load the original dataset
eng_data_original = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")
zgh_data_original = load_dataset("openlanguagedata/flores_plus", "zgh_Tfng", split="devtest")

data_original = pd.DataFrame({
    "ID": eng_data_original["id"],
    "eng": eng_data_original["text"],
    "zgh": zgh_data_original["text"],
})

data_original.describe(include='all')

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

,ID,eng,zgh
count,1012.000000,1012,1012
unique,NaN,1012,1012
top,NaN,Workers must often get their superiors' approv...,ⵉⴳⴳⵓⵜ ⵎⴰⴳ ⴷ ⵉⵇⵇⴰⵏ ⴰⴷ ⵢⵉⵍⵉ ⵓⵎⵙⴰⵙⴰ ⵏ ⵉⵏⵙⵙⵉⵅⴼⵏ ⵏ ...
freq,NaN,1,1
mean,505.500000,NaN,NaN
std,292.283538,NaN,NaN
min,0.000000,NaN,NaN
25%,252.750000,NaN,NaN
50%,505.500000,NaN,NaN
75%,758.250000,NaN,NaN


In [4]:
# sanity check
assert data_original['eng'].equals(data_revised['eng']), "English texts do not match!"
assert not data_original['zgh'].equals(data_revised['zgh']), "Revised and original Tamazight texts should not match!"

# Preprocess data

In [5]:
char = 'ⵒ'

# Check for the presence of the character in the revised dataset
print(data_revised['zgh'].apply(lambda x: char in x).any())

data_revised[data_revised['zgh'].str.contains(char)]['zgh']

True


207    ⵛⴰⵔⵏ ⵖ ⵓⵙⵉⵏⴰⴳ ⵏ ⵜⴰⵣⵔⴼⵜ ⴷ ⴷⵉⵎⵓⵇⵔⴰⵜⵉⵢⴰ ⵉ ⵜⵉⵖⵔⵉⵡⵉ...
309    ⵉⴽⵔⴼ ⵢⴰⵏ ⵉⴽⴰⵜⵉⵏ ⵉⴳⴰ ⴰⴱⵓⵍⵉⵙⵉ ⴰⴼⵉⵍⵉⵒⵉⵏⵉ ⵙⵉⵏ ⵉⵎⵍⵓ...
312    ⵟⵍⵇⵏ ⵉⵚⴹⵉⵙ ⵏ ⵔⴰⵀⴰⵢⵏ, ⵏⴳⵔⴰⵜⵙⵏ ⵜⴰⵣⴰⵏⵉⵏ ⴷⵉⵛⵉⴱⴰⵏⵏ,...
Name: zgh, dtype: object

In [6]:
# this code is adapted from  the Stopes repo of the NLLB team
# https://github.com/facebookresearch/stopes/blob/main/stopes/pipelines/monolingual/monolingual_line_processor.py#L214

import re
import sys
import typing as tp
import unicodedata
from sacremoses import MosesPunctNormalizer


mpn = MosesPunctNormalizer(lang="en")
mpn.substitutions = [
    (re.compile(r), sub) for r, sub in mpn.substitutions
]


def get_non_printing_char_replacer(replace_by: str = " ") -> tp.Callable[[str], str]:
    non_printable_map = {
        ord(c): replace_by
        for c in (chr(i) for i in range(sys.maxunicode + 1))
        # same as \p{C} in perl
        # see https://www.unicode.org/reports/tr44/#General_Category_Values
        if unicodedata.category(c) in {"C", "Cc", "Cf", "Cs", "Co", "Cn"}
    }

    def replace_non_printing_char(line) -> str:
        return line.translate(non_printable_map)

    return replace_non_printing_char

replace_nonprint = get_non_printing_char_replacer(" ")


def clean_text(text):
    clean = mpn.normalize(text)
    clean = replace_nonprint(clean)
    # replace 𝓕𝔯𝔞𝔫𝔠𝔢𝔰𝔠𝔞 by Francesca
    clean = unicodedata.normalize("NFKC", clean)
    return clean


def preprocess(text):
    text = text.replace('ⵒ', 'ⴱ').replace('ⵁ', 'ⵀ').replace('ⴴ', 'ⵖ')
    text = clean_text(text)
    return text

In [7]:
# apply preprocessing
data_revised['eng'] = data_revised['eng'].apply(preprocess)
data_revised['zgh'] = data_revised['zgh'].apply(preprocess)

data_original['eng'] = data_original['eng'].apply(preprocess)
data_original['zgh'] = data_original['zgh'].apply(preprocess)

In [8]:
char = 'ⵒ'

# Check for the presence of the character in the revised dataset
print(data_revised['zgh'].apply(lambda x: char in x).any())

data_revised[data_revised['zgh'].str.contains(char, na=False)]['zgh']

False


Series([], Name: zgh, dtype: object)

# Evaluate

In [9]:
eng_labels_revised = data_revised['eng'].tolist()
zgh_labels_revised = data_revised['zgh'].tolist()
eng_labels_original = data_original['eng'].tolist()
zgh_labels_original = data_original['zgh'].tolist()

In [10]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
ter_metric = evaluate.load("ter")

def metrics_calc(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    bleu_result = bleu_metric.compute(predictions = preds, references = labels)
    spm_result = bleu_metric.compute(predictions = preds, references = labels, tokenize='flores101')
    chrf_result = chrf_metric.compute(predictions = preds, references = labels, word_order=2)
    ter_result = ter_metric.compute(predictions = preds, references = labels)

    result = {
        'bleu': bleu_result['score'],
        'spbleu': spm_result['score'],
        'chrf++': chrf_result['score'],
        'ter': ter_result['score']
    }
    result = {k: round(v, 2) for k, v in result.items()}

    return result

In [11]:
import os

def evaluate_model(model, version):
    all_metrics = {}
    for direction in ['eng-zgh', 'zgh-eng']:
        if version == 'revised' and direction == 'eng-zgh':
            labels = zgh_labels_revised
        elif version == 'revised' and direction == 'zgh-eng':
            labels = eng_labels_revised
        elif version == 'original' and direction == 'zgh-eng':
            labels = eng_labels_original
        elif version == 'original' and direction == 'eng-zgh':
            labels = zgh_labels_original

        if version == 'original' and direction == 'eng-zgh': # the English references are the same in both datasets
            model_pred_path = os.path.join(PREDICTIONS_DIR, f"{model}-eng-zgh-revised.csv")
        else:
            model_pred_path = os.path.join(PREDICTIONS_DIR, f"{model}-{direction}-{version}.csv")
        df = pd.read_csv(model_pred_path)

        n_missing = df['prediction'].isnull().sum()
        if n_missing > 0:
            print(f"Warning: {n_missing} missing values found in {model_pred_path}. Filling with empty strings.")
            # replace missing values with empty strings
            df.fillna('', inplace=True)

        preds = df['prediction']
        preds = preds.apply(preprocess)
        preds = preds.tolist()

        metrics = metrics_calc(preds, labels)

        # add model and version info to metrics
        for key in list(metrics.keys()):
            metrics[f"{key}_{direction}"] = metrics.pop(key)
        all_metrics.update(metrics)

    return all_metrics

In [12]:
MODELS = ['nllb-600m',
          'nllb-600m-seed',
          'nllb-600m-all-data-1_25-epoch',
          'nllb-600m-all-data-1_25-epoch-2-beam',
          'nllb-600m-all-data-1_25-epoch-4-beam',
          'nllb-600m-all-data-1_25-epoch-8-beam',
          'nllb-distilled-1.3b',
          'nllb-1.3b',
          'nllb-3.3b',
          'gemini-2.5-pro',
          'claude-3.5-sonnet-20241022',
          'claude-3.7-sonnet-20250219'
          ]
PREDICTIONS_DIR = './predictions'

In [13]:
from tqdm import tqdm

results_revised = []

for model in tqdm(MODELS):
    result = evaluate_model(model, 'revised')
    results_revised.append(result)

results_revised_df = pd.DataFrame(results_revised, index=MODELS)

 75%|███████▌  | 9/12 [01:40<00:33, 11.20s/it]

100%|██████████| 12/12 [02:09<00:00, 10.79s/it]


In [14]:
order = ['bleu_eng-zgh', 'bleu_zgh-eng', 'spbleu_eng-zgh', 'spbleu_zgh-eng',
         'chrf++_eng-zgh', 'chrf++_zgh-eng', 'ter_eng-zgh', 'ter_zgh-eng']
results_revised_df = results_revised_df[order]
results_revised_df

,bleu_eng-zgh,bleu_zgh-eng,spbleu_eng-zgh,spbleu_zgh-eng,chrf++_eng-zgh,chrf++_zgh-eng,ter_eng-zgh,ter_zgh-eng
nllb-600m,4.96,15.30,19.29,15.86,25.64,37.38,94.77,76.06
nllb-600m-seed,8.01,17.19,25.70,17.74,31.69,39.70,83.32,73.01
nllb-600m-all-data-1_25-epoch,8.84,18.29,26.95,18.51,32.71,40.66,81.87,71.18
nllb-600m-all-data-1_25-epoch-2-beam,9.38,18.58,27.69,18.91,33.52,40.87,80.97,70.45
nllb-600m-all-data-1_25-epoch-4-beam,9.38,18.60,27.70,18.88,33.75,40.84,81.84,70.08
nllb-600m-all-data-1_25-epoch-8-beam,9.69,18.72,28.04,18.96,34.05,40.91,81.49,70.38
nllb-distilled-1.3b,7.02,18.32,23.35,18.83,29.07,39.97,85.99,71.68
nllb-1.3b,6.37,17.35,21.89,17.97,27.44,39.33,89.59,72.85
nllb-3.3b,7.68,18.05,24.92,18.82,29.80,39.80,83.42,72.14
gemini-2.5-pro,4.83,21.31,18.95,23.43,26.56,44.29,86.27,69.23


In [15]:
results_original = []

for model in tqdm(MODELS):
    result = evaluate_model(model, 'original')
    results_original.append(result)

results_original_df = pd.DataFrame(results_original, index=MODELS)

 75%|███████▌  | 9/12 [01:45<00:35, 11.93s/it]

100%|██████████| 12/12 [02:16<00:00, 11.41s/it]


In [16]:
results_original_df = results_original_df[order]
results_original_df

,bleu_eng-zgh,bleu_zgh-eng,spbleu_eng-zgh,spbleu_zgh-eng,chrf++_eng-zgh,chrf++_zgh-eng,ter_eng-zgh,ter_zgh-eng
nllb-600m,4.86,14.84,19.12,15.49,25.52,37.10,94.97,77.27
nllb-600m-seed,7.80,17.13,25.39,17.50,31.46,39.51,83.72,73.00
nllb-600m-all-data-1_25-epoch,8.64,17.97,26.66,18.20,32.50,40.27,82.17,71.72
nllb-600m-all-data-1_25-epoch-2-beam,9.12,18.29,27.32,18.53,33.26,40.45,81.38,70.71
nllb-600m-all-data-1_25-epoch-4-beam,9.15,18.32,27.33,18.55,33.48,40.56,82.30,70.47
nllb-600m-all-data-1_25-epoch-8-beam,9.42,18.37,27.67,18.55,33.79,40.55,81.92,70.63
nllb-distilled-1.3b,6.84,18.16,23.12,18.59,28.93,39.73,86.24,72.05
nllb-1.3b,6.34,17.23,21.80,17.79,27.35,39.11,89.75,73.28
nllb-3.3b,7.52,17.72,24.78,18.49,29.70,39.43,83.52,72.81
gemini-2.5-pro,4.70,21.34,18.70,23.37,26.40,44.35,86.59,69.16


In [17]:
# get the best performing model for each metric
print(results_revised_df.drop(columns=['ter_eng-zgh', 'ter_zgh-eng']).idxmax())
print(results_revised_df[['ter_eng-zgh', 'ter_zgh-eng']].idxmin())

bleu_eng-zgh      nllb-600m-all-data-1_25-epoch-8-beam
bleu_zgh-eng                            gemini-2.5-pro
spbleu_eng-zgh    nllb-600m-all-data-1_25-epoch-8-beam
spbleu_zgh-eng                          gemini-2.5-pro
chrf++_eng-zgh    nllb-600m-all-data-1_25-epoch-8-beam
chrf++_zgh-eng                          gemini-2.5-pro
dtype: object
ter_eng-zgh    nllb-600m-all-data-1_25-epoch-2-beam
ter_zgh-eng                          gemini-2.5-pro
dtype: object


In [18]:
# drop beam search results
beam_search_results = ['nllb-600m-all-data-1_25-epoch-2-beam',
    'nllb-600m-all-data-1_25-epoch-4-beam',
    'nllb-600m-all-data-1_25-epoch-8-beam'
    ]

results_revised_df = results_revised_df.drop(index=beam_search_results)
results_original_df = results_original_df.drop(index=beam_search_results)

In [19]:
diff = results_revised_df - results_original_df

In [20]:
diff['chrf++_eng-zgh'].mean().round(2)

np.float64(0.14)

In [21]:
diff['chrf++_zgh-eng'].mean().round(2)

np.float64(0.23)

In [22]:
[results_revised_df.loc['nllb-600m-seed'] - results_revised_df.loc['nllb-600m']]

[bleu_eng-zgh       3.05
 bleu_zgh-eng       1.89
 spbleu_eng-zgh     6.41
 spbleu_zgh-eng     1.88
 chrf++_eng-zgh     6.05
 chrf++_zgh-eng     2.32
 ter_eng-zgh      -11.45
 ter_zgh-eng       -3.05
 dtype: float64]